# IOAI — 2025 Stage 1 Hidden Subsequences (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/test.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-stage-1-hidden-subsequences/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 숨은 부분열 — 베이스라인 (선형 회귀)

이진 시퀀스를 그대로 입력해 값을 선형 회귀한다. 부분열(순서 있는, 비연속) 논리를 못 담아 MSE 가 크다(≈200 → 0점). 전체 원문은 Overview 탭 참고.

> 참고: 원본 Google Drive 데이터는 접근 제한(다운로드 쿼터)이라, 이 사이트는 **동일한 생성 규칙**(숨은 패턴 3개+정수 가중치의 부분열 가중합)으로 만든 등가 데이터를 씁니다.

## 데이터 로드

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"; print(device)
L = 20; cols = [f"s{i}" for i in range(L)]
tr = pd.read_csv("data/train_dataset.csv")
Xtr = torch.tensor(tr[cols].values, dtype=torch.long)
ytr = torch.tensor(tr["value"].values, dtype=torch.float32).unsqueeze(1)
te = pd.read_csv("data/test.csv")
Xte = torch.tensor(te[cols].values, dtype=torch.long).to(device)
test_ids = te["id"].tolist()
dl = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xtr, ytr), batch_size=128, shuffle=True)
print("train", Xtr.shape, "test", Xte.shape)

## 선형 모델

In [ ]:
class YourModel(nn.Module):
    def __init__(self, L):
        super().__init__(); self.layer = nn.Linear(L, 1)     # 시퀀스를 그대로 선형 회귀
    def forward(self, x): return self.layer(x.float())
model = YourModel(L).to(device)

## 학습(≤4000 iter)

In [ ]:
opt = torch.optim.Adam(model.parameters(), 0.01); crit = nn.MSELoss()
model.train(); it = iter(dl)
for step in range(4000):                       # 과제 제약: 최대 4000 iteration
    try: xb, yb = next(it)
    except StopIteration: it = iter(dl); xb, yb = next(it)
    xb, yb = xb.to(device), yb.to(device)
    opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
print("params:", sum(p.numel() for p in model.parameters()), "(<50000 required)")

## 예측 → submission.csv

In [ ]:
model.eval()
with torch.no_grad(): pred = model(Xte).squeeze(1).cpu().numpy()
pd.DataFrame({"id": test_ids, "value": np.rint(pred).astype(int)}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(pred))

순서를 보는 모델(Embedding+BiLSTM)이 부분열 논리를 훨씬 잘 잡는다 — 모범답안 참고.

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)